# RSNA Knee MRI — Mini Set 预处理（Kaggle Notebook）

> **阶段一配置**：384 分辨率 + Sagittal 主序列 + 16 张中间切片
>
> 一轮跑完，输出 ~2-3 GB，直接下载。

## 0. 配置参数

In [ ]:
TARGET_SIZE = 384              # 阶段一：256~384 足够
MAX_SLICES_PER_SERIES = 16     # 每序列取中间 16 张（丢弃两端噪声/定位像）
MINI_TOTAL = 600              # Mini 集目标 exam 数
SEED = 2026

import numpy as np
import pandas as pd
import pydicom
import cv2
import tarfile
import io
import shutil
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# 加速 DICOM 解码
try:
    pydicom.config.image_handlers = ['gdcm', 'pillow', 'jpeg_ls']
except Exception:
    pass

COMP_DIR = Path('/kaggle/input/rsna-knee-abnormality-detection')
WORK_DIR = Path('/kaggle/working')

bytes_per_slice = TARGET_SIZE * TARGET_SIZE * 2  # float16
print(f'分辨率: {TARGET_SIZE}  |  每序列 {MAX_SLICES_PER_SERIES} 张切片')
print(f'预估单切片: {bytes_per_slice/1024:.0f} KB')
print(f'预估总输出: {600 * MAX_SLICES_PER_SERIES * bytes_per_slice / 1024**3 * 0.65:.0f} GB 压缩')

## 1. 加载元数据 & 采样

In [ ]:
TARGET_COLS = [
    'ACL', 'MCL',
    'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA',
    'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture'
]

train = pd.read_csv(COMP_DIR / 'train.csv')
series = pd.read_csv(COMP_DIR / 'train_series.csv')

for col in TARGET_COLS:
    train[col] = pd.to_numeric(train[col], errors='coerce').fillna(0).astype(int)

train['n_labels'] = train[TARGET_COLS].sum(axis=1)
pos_exams = train[train['n_labels'] > 0]
neg_exams = train[train['n_labels'] == 0]

np.random.seed(SEED)
neg_selected = neg_exams.sample(n=MINI_TOTAL - len(pos_exams), random_state=SEED)
mini_set = pd.concat([pos_exams, neg_selected], ignore_index=True)

print(f'Mini 集: {len(mini_set)} exams ({len(pos_exams)} 异常 + {len(neg_selected)} 正常)')

## 2. 匹配主序列（三级降级）

阶段一只用 **每 exam 一个主序列**：优先 Sagittal FS Fluid → Sagittal → 任意。

In [ ]:
tier1 = series[
    (series['Anatomical_Plane'] == 'Sagittal')
    & (series['Fat_Suppression'] == 1)
    & (series['Fluid_Sensitive'] == 1)
]
tier2 = series[series['Anatomical_Plane'] == 'Sagittal']

results = []
tier_counts = {1: 0, 2: 0, 3: 0}

for _, row in mini_set.iterrows():
    sid = row['StudyInstanceUID']
    p = tier1[tier1['StudyInstanceUID'] == sid]
    tier = 1
    if len(p) == 0:
        p = tier2[tier2['StudyInstanceUID'] == sid]
        tier = 2
    if len(p) == 0:
        p = series[series['StudyInstanceUID'] == sid].head(1)
        tier = 3
    for _, sr in p.iterrows():
        results.append({
            'StudyInstanceUID': sid,
            'SeriesInstanceUID': sr['SeriesInstanceUID'],
            'tier': tier,
            'plane': sr.get('Anatomical_Plane', 'Unknown'),
        })
        tier_counts[tier] += 1

download_list = pd.DataFrame(results)
print(f'Tier 1 (Sagittal+FS+Fluid): {tier_counts[1]}')
print(f'Tier 2 (任何 Sagittal):     {tier_counts[2]}')
print(f'Tier 3 (Fallback):           {tier_counts[3]}')
print(f'总序列: {len(download_list)}  |  涉及 exam: {download_list["StudyInstanceUID"].nunique()}')

## 3. 定位 DICOM 文件

In [ ]:
train_series_root = COMP_DIR / 'train_series'

study_to_series = defaultdict(list)
for _, row in download_list.iterrows():
    study_to_series[row['StudyInstanceUID']].append(row['SeriesInstanceUID'])

series_groups = defaultdict(list)  # {(study, series): [dcm_paths]}

for study_uid, series_uids in tqdm(study_to_series.items(), desc='定位 DICOM'):
    study_dir = train_series_root / study_uid
    if not study_dir.is_dir():
        continue
    for series_uid in series_uids:
        series_dir = study_dir / series_uid
        if series_dir.is_dir():
            dcms = sorted(series_dir.glob('*.dcm'))
            if dcms:
                series_groups[(study_uid, series_uid)] = [str(d) for d in dcms]

n_series = len(series_groups)
n_dcms = sum(len(v) for v in series_groups.values())
print(f'系列: {n_series}  |  DICOM: {n_dcms}  |  每系列平均: {n_dcms/n_series:.0f} 切片')

## 4. 预处理 → 流式写入 tar.gz

In [ ]:
def preprocess_dicom(dcm_path, target_size):
    """DICOM -> float16 (target_size, target_size)"""
    ds = pydicom.dcmread(dcm_path, stop_before_pixels=False)
    img = ds.pixel_array.astype(np.float32)
    p_low, p_high = np.percentile(img, [0.5, 99.5])
    if p_high > p_low:
        img = np.clip((img - p_low) / (p_high - p_low), 0.0, 1.0)
    else:
        img = np.zeros_like(img)
    h, w = img.shape
    size = min(h, w)
    h_start = (h - size) // 2
    w_start = (w - size) // 2
    img = img[h_start:h_start+size, w_start:w_start+size]
    if size != target_size:
        img = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_LINEAR)
    return img.astype(np.float16)

print('预处理函数就绪')

In [ ]:
archive_path = WORK_DIR / 'mini_set_npy.tar.gz'
meta_rows = []
total_slices = 0
errors = 0

with tarfile.open(archive_path, 'w:gz') as tar:
    for (study_uid, series_uid), dcm_list in tqdm(
        series_groups.items(), desc='预处理'
    ):
        n = len(dcm_list)
        if n <= MAX_SLICES_PER_SERIES:
            selected = dcm_list
        else:
            start = (n - MAX_SLICES_PER_SERIES) // 2
            selected = dcm_list[start:start + MAX_SLICES_PER_SERIES]
        
        for dcm_path in selected:
            try:
                sop_uid = Path(dcm_path).stem
                img = preprocess_dicom(dcm_path, TARGET_SIZE)
                
                buf = io.BytesIO()
                np.save(buf, img)
                buf.seek(0)
                data = buf.getvalue()
                
                arcname = f'npy/{study_uid}/{series_uid}/{sop_uid}.npy'
                info = tarfile.TarInfo(name=arcname)
                info.size = len(data)
                tar.addfile(info, io.BytesIO(data))
                
                total_slices += 1
                meta_rows.append({
                    'StudyInstanceUID': study_uid,
                    'SeriesInstanceUID': series_uid,
                    'SOPInstanceUID': sop_uid,
                })
            except Exception as e:
                errors += 1
                if errors <= 10:
                    print(f'  Error [{Path(dcm_path).name[:40]}]: {e}')

tar_size = archive_path.stat().st_size / 1024**3

meta_df = pd.DataFrame(meta_rows)
meta_csv = WORK_DIR / 'mini_set_metadata.csv'
meta_df.to_csv(meta_csv, index=False)

print(f'\n===== 完成 =====')
print(f'切片: {total_slices}  |  错误: {errors}')
print(f'输出: {archive_path.name} ({tar_size:.1f} GB)')
print(f'元数据: {meta_csv.name} ({len(meta_df)} 行)')
print(f'涉及 exam: {meta_df["StudyInstanceUID"].nunique()}')
print(f'涉及序列: {meta_df["SeriesInstanceUID"].nunique()}')

## 5. 下载 & 解压

右侧 **Output** 面板 → 下载 `mini_set_npy.tar.gz` 和 `mini_set_metadata.csv` → 放到 `RSNA/data/`

```powershell
cd RSNA\data
tar -xzf mini_set_npy.tar.gz
# 得到: data/npy/{StudyInstanceUID}/{SeriesInstanceUID}/{SOPInstanceUID}.npy
#      data/mini_set_metadata.csv
```